# LOOK — Observe activations (cell by cell)

Same code as `scripts/look.py`, but one **mode per cell** so you can read the output slowly.

**Kernel:** `CXR local Qwen (faiss_gpu1)`  
**Observe only** — no steering, no patching.

Run order:
1. **Setup** (imports)
2. **Load model** (once; ~1 min first time)
3. Any mode cell below, in any order

## Setup

In [ ]:
import json
import os
import sys
from pathlib import Path

SCRIPTS = Path("/home/udonsi-kalu/staging/cxr-mi-repeng-grounding/scripts")
CASES = Path("/home/udonsi-kalu/staging/cxr-mi-repeng-grounding/learning_lab/cases/oncology_m1.json")

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

import look
from _common import PROMPT_A, PROMPT_B, PROMPT_TEST, DEFAULT_MODEL, boot, encode, last_residual, device

cases = json.loads(CASES.read_text(encoding="utf-8"))
NOTE = cases["foundation_note"]
LAYER = 20

print("Teaching note:", NOTE)
print("Built-in A/B/test prompts loaded.")


## Load model (run once)

In [2]:
model, tok = boot(DEFAULT_MODEL, layer=LAYER)

load Qwen/Qwen2.5-7B-Instruct (local only) …


Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.07it/s]
Some parameters are on the meta device because they were offloaded to the cpu.


blocks=28  hidden=3584  using layer 20


---
## Mode: `capture`
Last-token hidden state at one layer — shape, L2, first 10 dims.

In [3]:
look.cmd_capture(model, tok, layer=LAYER, text=NOTE)

prompt tokens: 15
shape last-token: (1, 15, 3584)  [printing last token only]
layer 20 last-token L2: 114.6233
first 10: [0.75879, -2.01758, 0.82471, 0.63623, -0.3064, -0.81738, 1.39551, -0.2207, -2.35547, -0.08154]


## Mode: `sites`
Three cameras on the same layer: block, attention, MLP.

In [4]:
look.cmd_sites(model, tok, layer=0, text=NOTE)

L0 block  dim=3584  L2=10.4349
L0 attn   dim=3584  L2=7.3859
L0 mlp    dim=3584  L2=4.5926


## Mode: `positions`
L2 at **every token** — see which position is last.

In [5]:
look.cmd_positions(model, tok, layer=LAYER, text=NOTE)

layer 20  tokens=15  dim=3584
  [  0] L2=15568.237  Patient
  [  1] L2=134.829  Ġreceived
  [  2] L2=117.631  ĠF
  [  3] L2=119.446  OL
  [  4] L2=140.427  FOX
  [  5] L2=129.358  .
  [  6] L2=133.687  ĠDisease
  [  7] L2=142.629  Ġprogressed
  [  8] L2=128.677  .
  [  9] L2=111.908  ĠF
  [ 10] L2=112.350  OL
  [ 11] L2=123.718  FOX
  [ 12] L2=132.923  Ġwas
  [ 13] L2=130.808  Ġdiscontinued
  [ 14] L2=114.623  .  <- last


## Mode: `layer_sweep`
Last-token L2 at several layers through the stack.

In [6]:
look.cmd_layer_sweep(model, tok, text=NOTE)

last-token L2 by layer for 'Patient received FOLFOX. Disease progressed. FOLFOX was discontinued.'
  L 0  L2= 10.435  dim=3584
  L 7  L2= 37.171  dim=3584
  L14  L2= 59.643  dim=3584
  L20  L2=114.623  dim=3584
  L21  L2=146.036  dim=3584
  L27  L2=671.217  dim=3584


## Mode: `difference`
Two built-in prompts (A vs B) → difference direction → where test prompt sits.

In [7]:
print("A:", PROMPT_A)
print("B:", PROMPT_B)
print("test:", PROMPT_TEST)
print()
look.cmd_difference(model, tok, layer=LAYER)

A: Progress note: FOLFOX completed last month. Patient is now on observation. Has FOLFOX stopped? Answer yes or no.
B: Progress note: FOLFOX completed last month. Patient continues FOLFOX this cycle. Has FOLFOX stopped? Answer yes or no.
test: Progress note: Oxaliplatin/5-FU course ended in March. Surveillance only. Has FOLFOX stopped? Answer yes or no.

layer 20  dim 3584
||h_A|| 110.8236  ||h_B|| 111.0922
||A-B|| 27.1845
cos to (A-B)  A=+0.1128  B=-0.1322  test=+0.0614
test closer to A if score > 0, to B if score < 0


## Mode: `logits`
Next-token logits after the test prompt.

In [8]:
look.cmd_logits(model, tok, PROMPT_TEST)

next-token logits (top 8):
   +25.719  ' No'
   +22.969  ' Based'
   +21.906  ' Yes'
   +21.000  ' To'
   +20.359  ' The'
   +19.828  ' If'
   +18.703  ' A'
   +18.656  ' Given'


## Mode: `logit_lens`
Decode last-token residual at several layers as if each were the final layer.

In [9]:
look.cmd_logit_lens(model, tok, PROMPT_TEST)

logit lens: decode last-token residual as if it were final
  L 0  ' strugg':14.20, ' undefeated':12.07, ' thems':11.61, '-bedroom':11.42, ' WHETHER':11.35
  L14  ' 若要':16.80, '换句话':16.05, 'achsen':14.57, "'email":14.45, ' ($("#':14.34
  L20  ' 若要':17.39, ' yes':16.98, ' Yes':14.83, '换句话':13.64, 'yes':13.40
  L27  ' No':25.72, ' Based':22.97, ' Yes':21.91, ' To':21.00, ' The':20.36


## Mode: `sae`
Project last-token residual through the Chanin L20 SAE (65536 features).

In [10]:
look.cmd_sae(model, tok, layer=LAYER)

SAE d_in=3584 d_sae=65536 nonzero=78
top feature ids: [(1307, 24.7989), (182, 24.5507), (719, 22.5684), (1196, 22.4065), (1306, 21.3416), (1877, 19.9181), (27224, 18.5181), (1099, 17.5924)]


---
## Bonus: keep a tensor in memory
Jupyter advantage — inspect `h` after the cell runs.

In [ ]:
ids = encode(tok, NOTE, device(model))
h = last_residual(model, ids, layer=LAYER)

print("shape:", tuple(h.shape))
print("L2:", float(h.norm()))
h[:10]